In [0]:
# Code Generated by Sidekick is for learning and experimentation purposes only.

# filename: zorder_sales_transactions.py
# ─────────────────────────────────────────────────────
# STEP 1: Imports + Spark Session + Logging
# ─────────────────────────────────────────────────────
import time
import logging
from datetime import datetime
from pyspark.sql import SparkSession

logging.basicConfig(
    level  = logging.INFO,
    format = "%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

spark = SparkSession.builder \
    .appName("TICKET-1001-ZOrder-Sales") \
    .getOrCreate()


# ─────────────────────────────────────────────────────
# STEP 2: Configuration
# ─────────────────────────────────────────────────────
TABLE_NAME     = "retailmart.sales_transactions"
ZORDER_COLUMNS = ["city", "product_category", "sale_date"]
BENCHMARK_QUERY = f"""
    SELECT city, product_category,
           COUNT(*) AS total_orders,
           SUM(amount) AS total_revenue,
           AVG(amount) AS avg_order_value
    FROM {TABLE_NAME}
    WHERE sale_date >= date_sub(current_date(), 90)
    AND   city             = 'New York'
    AND   product_category = 'Electronics'
    GROUP BY city, product_category
    ORDER BY total_revenue DESC
"""


# ─────────────────────────────────────────────────────
# STEP 3: Benchmark Function
# ─────────────────────────────────────────────────────
def run_benchmark_query(label: str) -> float:
    logger.info(f"[{label}] Running benchmark query...")
    start_time = time.time()
    spark.sql(BENCHMARK_QUERY).collect()
    elapsed = round(time.time() - start_time, 2)
    logger.info(f"[{label}] Query completed in {elapsed}s")
    return elapsed


# ─────────────────────────────────────────────────────
# STEP 4: Table Stats Function
# ─────────────────────────────────────────────────────
def get_table_stats(label: str) -> dict:
    logger.info(f"[{label}] Collecting table stats...")
    stats_row = spark.sql(f"""
        DESCRIBE DETAIL {TABLE_NAME}
    """).select("numFiles", "sizeInBytes").collect()[0]

    stats = {
        "num_files"    : stats_row["numFiles"],
        "size_in_bytes": stats_row["sizeInBytes"],
        "size_in_mb"   : round(stats_row["sizeInBytes"] / (1024 * 1024), 2)
    }
    logger.info(
        f"[{label}] Files: {stats['num_files']} | "
        f"Size: {stats['size_in_mb']} MB"
    )
    return stats


# ─────────────────────────────────────────────────────
# STEP 5: Main Optimization Function
# ─────────────────────────────────────────────────────
def run_optimization() -> dict:
    # PHASE 1: Before metrics
    logger.info("PHASE 1: Capturing BEFORE metrics")
    stats_before      = get_table_stats("BEFORE")
    query_time_before = run_benchmark_query("BEFORE")

    # PHASE 2: Run OPTIMIZE + ZORDER
    logger.info("PHASE 2: Running OPTIMIZE + ZORDER BY")
    zorder_cols_str = ", ".join(ZORDER_COLUMNS)
    try:
        optimize_start = time.time()
        spark.sql(f"""
            OPTIMIZE {TABLE_NAME}
            ZORDER BY ({zorder_cols_str})
        """)
        logger.info(
            f"✅ OPTIMIZE completed in "
            f"{round(time.time() - optimize_start, 2)}s"
        )
    except Exception as e:
        logger.error(f"❌ OPTIMIZE failed: {str(e)}")
        raise

    # PHASE 3: After metrics
    logger.info("PHASE 3: Capturing AFTER metrics")
    stats_after      = get_table_stats("AFTER")
    query_time_after = run_benchmark_query("AFTER")

    # PHASE 4: Calculate improvement
    query_improvement = round(
        ((query_time_before - query_time_after)
         / query_time_before) * 100, 1
    )
    file_reduction = round(
        ((stats_before["num_files"] - stats_after["num_files"])
         / stats_before["num_files"]) * 100, 1
    )

    # PHASE 5: Build summary
    return {
        "ticket"               : "TICKET-1001",
        "table"                : TABLE_NAME,
        "optimized_at"         : datetime.now().isoformat(),
        "zorder_columns"       : ZORDER_COLUMNS,
        "files_before"         : stats_before["num_files"],
        "files_after"          : stats_after["num_files"],
        "file_reduction_pct"   : file_reduction,
        "query_time_before"    : query_time_before,
        "query_time_after"     : query_time_after,
        "query_improvement_pct": query_improvement,
        "status"               : "SUCCESS"
    }


# ─────────────────────────────────────────────────────
# STEP 6: Entry Point
# ─────────────────────────────────────────────────────
if __name__ == "__main__":
    logger.info("🚀 Starting TICKET-1001 Z-Order Optimization")
    result = run_optimization()

    logger.info("=" * 55)
    logger.info("OPTIMIZATION COMPLETE — FINAL SUMMARY")
    logger.info("=" * 55)
    logger.info(f"Files    : {result['files_before']} → {result['files_after']} ({result['file_reduction_pct']}% reduction)")
    logger.info(f"Query    : {result['query_time_before']}s → {result['query_time_after']}s ({result['query_improvement_pct']}% faster)")
    logger.info(f"Status   : {result['status']} ✅")
    logger.info("=" * 55)



In [0]:
from datetime import date

today_str=date.today().isoformat()

history=spark.sql(f"""
    DESCRIBE HISTORY {TABLE_NAME}
""").filter("operation = 'OPTIMIZE'"
            ).filter(f"DATE(timestamp)='{today_str}'").collect()

# check if any of today's optimize z-order columns
for row in history:
    operation_params=str(row['operationParameters'])
    if "zOrderBy" in operation_params:
        zorder_value=row["operationParameters"].get("zOrderBy","[]")
        if zorder_value!="[]" and zorder_value!="":
            logger.info(f"""
                        Zorder already ran today at {row["timestamp"]} with columns {zorder_value}
                        """)
            break
        else:
            logger.info(f"""
                        Running Zorder today at {datetime.now().isoformat()}
                        """)